# Smart Audio Notes: Detect, Summarize, Ask (CPU only)

Whisper listens. Gemma thinks. Both are open source and run on CPU.

**What this notebook does:**
1. Turns your audio into text
2. Detects the type of audio by itself (lecture, meeting, casual talk, voice note, shopping, recipe, and more)
3. Writes a summary that fits that type
4. Lets you ask any question about the audio. Key points, tasks, exam questions, ingredients, shopping list: just ask.

Set **Runtime > Change runtime type** to **CPU**.

## 1. Install

In [ ]:
!pip install -q faster-whisper huggingface_hub
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu

## 2. Fix for the av error and upload audio

In [ ]:
import av

# Safe patch: ignore the metadata_errors parameter.
# The original function is saved only once, so running this cell again is safe.
if not hasattr(av, "_original_open"):
    av._original_open = av.open

def patched_open(*args, **kwargs):
    kwargs.pop("metadata_errors", None)
    return av._original_open(*args, **kwargs)

av.open = patched_open

from google.colab import files
uploaded = files.upload()          # choose your audio file
AUDIO_PATH = "/content/" + list(uploaded.keys())[0]
print("Using file:", AUDIO_PATH)

## 3. Audio to text

Set `LANGUAGE` to `"en"`, `"hi"` or `"bn"`. Use `None` to let the model guess.

In [ ]:
import time
from faster_whisper import WhisperModel

LANGUAGE = None   # "en", "hi", "bn" or None

t0 = time.time()
whisper = WhisperModel("base", device="cpu", compute_type="int8", cpu_threads=2)
segments, info = whisper.transcribe(AUDIO_PATH, vad_filter=True, language=LANGUAGE)

print("Language:", info.language)
print("Audio length (sec):", round(info.duration))

lines = []
for s in segments:
    lines.append(s.text.strip())

transcript = " ".join(lines)
print("Done in", round(time.time() - t0), "seconds")
print("Words:", len(transcript.split()))
print("\nPreview:\n", transcript[:500])

## 4. Load Gemma 3 1B

This downloads a small file (about 800 MB). It is a 4 bit version, so it fits in 8GB RAM.

In [ ]:
from llama_cpp import Llama

llm = Llama.from_pretrained(
    repo_id="unsloth/gemma-3-1b-it-GGUF",
    filename="gemma-3-1b-it-Q4_K_M.gguf",
    n_ctx=4096,
    n_threads=2,
    verbose=False,
)

def ask(prompt, max_tokens=400):
    out = llm.create_chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=0.2,
    )
    return out["choices"][0]["message"]["content"].strip()

print(ask("Say hello in one short sentence."))

## 5. Detect the audio type and summarize

First, Gemma picks the type. Then the summary prompt changes to fit that type.
For long audio, the text is cut into parts, each part is summarized, then the summaries are joined.

In [ ]:
TYPES = {
    "lecture": "This is a lecture or class. Write the main topic, then the main ideas that were taught.",
    "meeting": "This is a meeting. Write the purpose, what was discussed, and any decisions made.",
    "casual talk": "This is a casual chat. Write who or what was talked about and the main points in a friendly way.",
    "voice note": "This is a personal voice note. Write the main message and anything the speaker wants to remember or do.",
    "shopping": "This is a shopping or shop conversation. Write what items, prices, quantities or deals were talked about.",
    "recipe": "This is a cooking recipe. Write the dish name, the ingredients, and the main steps in order.",
    "phone call": "This is a phone call. Write why the call happened and what was agreed.",
    "interview": "This is an interview. Write who was asked about what, and the main answers.",
    "story or news": "This is a story or news. Write what happened, to whom, and where or when.",
    "other": "Write what the audio is about and the main points.",
}

RULES = "Use simple English and short sentences. Use only the text given. Do not invent facts."

def split_words(text, size):
    words = text.split()
    return [" ".join(words[i:i+size]) for i in range(0, len(words), size)]

def detect_type(text):
    sample = " ".join(text.split()[:350])
    options = ", ".join(TYPES.keys())
    reply = ask(
        "Read this text from an audio recording. Which type is it? "
        "Choose exactly one from this list: " + options + ". "
        "Reply with only the type name.\n\nTEXT:\n" + sample,
        max_tokens=10,
    ).lower()
    for t in TYPES:
        if t in reply:
            return t
    return "other"

audio_type = detect_type(transcript)
print("Detected type:", audio_type.upper())

type_hint = TYPES[audio_type]

parts = split_words(transcript, 500)
print("Parts:", len(parts))

if len(parts) == 1:
    summary = ask(
        "Summarize this audio in one short paragraph (4 to 7 sentences). "
        + type_hint + " " + RULES + "\n\nTEXT:\n" + parts[0],
        max_tokens=350,
    )
else:
    part_notes = []
    for i, p in enumerate(parts, 1):
        print("Summarizing part", i, "of", len(parts), "...")
        part_notes.append(ask(
            "Summarize this part in 3 to 5 short bullet points. "
            + type_hint + " " + RULES + "\n\nTEXT:\n" + p
        ))
    joined = "\n\n".join(part_notes)
    # keep the joined notes inside the context window
    joined = " ".join(joined.split()[:1800])
    summary = ask(
        "These are notes from one audio recording. Write one short summary paragraph "
        "(5 to 7 sentences). " + type_hint + " " + RULES + "\n\nNOTES:\n" + joined,
        max_tokens=350,
    )

print("\nSUMMARY\n", summary)

## 6. Prepare the content window

Gemma 1B can read only about 4000 tokens at one time. So we cannot send a long transcript in one go.
This cell cuts the text into small chunks. When you ask a question, the tool finds the 3 chunks that match your question best and sends only those.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

CHUNK_WORDS = 250
TOP_K = 3

chunks = split_words(transcript, CHUNK_WORDS)
print("Chunks:", len(chunks))

# token pattern works for English, Hindi and Bengali
vec = TfidfVectorizer(token_pattern=r"[^\s.,!?;:()]+", sublinear_tf=True)
chunk_matrix = vec.fit_transform(chunks)

def get_context(question, k=TOP_K):
    if len(chunks) <= k:
        return "\n\n".join(chunks)
    sims = cosine_similarity(vec.transform([question]), chunk_matrix)[0]
    best = sorted(np.argsort(sims)[-k:])      # keep original order
    return "\n\n".join(chunks[i] for i in best)

def answer_question(question):
    context = get_context(question)
    prompt = (
        "You help a person understand an audio recording. "
        "The audio type is: " + audio_type + ".\n"
        "Answer the question using only the SUMMARY and the EXCERPTS below. "
        "If the answer is not there, say: I did not hear this in the audio. "
        "Answer in the same language as the question. " + RULES + "\n\n"
        "SUMMARY:\n" + summary + "\n\n"
        "EXCERPTS:\n" + context + "\n\n"
        "QUESTION: " + question
    )
    return ask(prompt, max_tokens=400)

## 7. Ask your questions

Type a question and press Enter. Type `exit` or press Enter on an empty line to stop.
You can run this cell again any time.

**Ideas to ask:**
- Give me the key points.
- Is there any homework or deadline?
- Make 5 exam questions with answers.
- Explain the hard words in simple language.
- What should I buy? (shopping)
- What are the ingredients and steps? (recipe)
- What did the speaker decide? (meeting)

In [ ]:
try:
    qa_log
except NameError:
    qa_log = []

print("Audio type:", audio_type.upper())
print("Ask your question. Type exit to stop.")

while True:
    q = input("\nYour question: ").strip()
    if q.lower() in ("", "exit", "quit", "q"):
        print("Stopped.")
        break
    a = answer_question(q)
    print("\nAnswer:\n", a)
    qa_log.append((q, a))

## 8. Save and download

In [ ]:
qa_text = "\n\n".join("Q: " + q + "\nA: " + a for q, a in qa_log) or "No questions asked."

report = f"""# Audio Notes

## Type
{audio_type}

## Summary
{summary}

## Questions and Answers
{qa_text}

## Full Transcript
{transcript}
"""

with open("audio_notes.md", "w", encoding="utf-8") as f:
    f.write(report)

files.download("audio_notes.md")

## Notes

- Gemma 3 1B can make mistakes. Check important answers against the transcript.
- If an answer says it did not hear something, try asking in different words. The search uses the words in your question.
- Very broad questions on long audio, like list everything, work best on the summary. Ask about one topic at a time for better answers.